# Section 3 - Generation + Evaluation



In this section, we evaluate how effectively our RAG system retrieves the correct NUS module for a user's question.

For each query, we specify the expected module code and compare it with the modules returned by our retrieval system.

We use two metrics:

- **Hit@5:** Measures the proportion of queries for which the correct module appears in the top five retrieved results.
- **Mean Reciprocal Rank (MRR):** Measures how highly the correct module is ranked, rewarding results that appear closer to the top.

These metrics help us assess retrieval performance before investigating potential improvements.

In [53]:
import sys
from pathlib import Path

# Go from notebooks/ to the week-11-nlp project root
workshop_dir = Path.cwd().parent

sys.path.insert(0, str(workshop_dir))

import rag_utils
import pandas as pd


In [54]:
if rag_utils.modules is None:
    rag_utils.load()

print("Loaded modules:", len(rag_utils.modules))
print("Embeddings shape:", rag_utils.embeddings.shape)

Loaded modules: 6552
Embeddings shape: (6552, 384)


In [55]:
eval_path = workshop_dir / "data" / "eval_set_v2.csv"

eval_df = pd.read_csv(eval_path)

print(f"Total evaluation queries: {len(eval_df)}")
print("\nQueries by category:")
display(eval_df["category"].value_counts().to_frame("count"))

display(eval_df.head())

Total evaluation queries: 40

Queries by category:


,count
category,
paraphrase,31
vague,5
keyword,2
prerequisite,2


,query,relevant,category
0,I want to study how computers process human la...,CS4248|DSA4213|CP4284,paraphrase
1,how do apps like Google Lens recognise what's ...,CS4243|EE4212|CS4277|EE4704,paraphrase
2,teach an AI agent to learn by trial and error ...,CS4246|MA4275,paraphrase
3,how to store and query data for a website back...,CS2102|IT2002|TCX2003|TIC2601,paraphrase
4,what happens inside my laptop when I run sever...,CS2106|CG2271,paraphrase


In [56]:
def evaluate_multilabel_retrieval(eval_df, k=5):
    details = []

    for _, row in eval_df.iterrows():
        query = str(row["query"])
        relevant_codes = {
            code.strip().upper()
            for code in str(row["relevant"]).split("|")
            if code.strip()
        }

        results = rag_utils.search(query, k=k)

        ranked_codes = [
            str(module.get("moduleCode", "")).upper()
            for module, score in results
        ]

        # Rank of the first retrieved relevant module
        relevant_ranks = [
            rank for rank, code in enumerate(ranked_codes, start=1)
            if code in relevant_codes
        ]

        first_relevant_rank = (
            min(relevant_ranks) if relevant_ranks else 0
        )

        details.append({
            "query": query,
            "category": row["category"],
            "relevant_codes": " | ".join(sorted(relevant_codes)),
            "top_k_module_codes": " | ".join(ranked_codes),
            "rank_of_first_relevant": first_relevant_rank,
            f"hit_at_{k}": int(bool(relevant_ranks)),
            "reciprocal_rank": (
                1 / first_relevant_rank
                if first_relevant_rank else 0
            )
        })

    details_df = pd.DataFrame(details)

    summary = {
        "n_queries": len(details_df),
        f"hit_at_{k}": details_df[f"hit_at_{k}"].mean(),
        "mrr": details_df["reciprocal_rank"].mean()
    }

    return summary, details_df


In [57]:
summary, details_df = evaluate_multilabel_retrieval(
    eval_df,
    k=5
)

print(f"Queries evaluated: {summary['n_queries']}")
print(f"Hit@5: {summary['hit_at_5']:.3f}")
print(f"MRR: {summary['mrr']:.3f}")

results_dir = workshop_dir / "evaluation" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "eval_set_v2_results.csv"
details_df.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")

display(details_df)

Queries evaluated: 40
Hit@5: 0.575
MRR: 0.405
Results saved to: /Users/dharshiniilangovan/Downloads/NLP-Workshop/week-11-nlp/evaluation/results/eval_set_v2_results.csv


,query,category,relevant_codes,top_k_module_codes,rank_of_first_relevant,hit_at_5,reciprocal_rank
0,I want to study how computers process human la...,paraphrase,CP4284 | CS4248 | DSA4213,HS2914 | PL3281E | EL2206 | EL1101E | EL3206,0,0,0.000000
1,how do apps like Google Lens recognise what's ...,paraphrase,CS4243 | CS4277 | EE4212 | EE4704,NST2050 | EE6733 | EE4212 | AIS5202 | CS5240,3,1,0.333333
2,teach an AI agent to learn by trial and error ...,paraphrase,CS4246 | MA4275,CS3227 | TCX2006 | IS4403 | EE2213 | MSI5001,0,0,0.000000
3,how to store and query data for a website back...,paraphrase,CS2102 | IT2002 | TCX2003 | TIC2601,TIC2601 | DSA5104 | CS3223 | IT2002 | TCX2003,1,1,1.000000
4,what happens inside my laptop when I run sever...,paraphrase,CG2271 | CS2106,CS6215 | CS5218 | CS5239 | CS3210 | CS2100DE,0,0,0.000000
5,how does code I write get translated into mach...,paraphrase,CS4212,GET1036 | CS4215 | CS4212 | GEX1019 | HS2914,3,1,0.333333
6,how do I keep messages secret so only the rece...,paraphrase,CS4230 | CS4236,CS4230 | CS5430 | SWE5005G | SWE5005 | CS3236,1,1,1.000000
7,I want to build my own video game,paraphrase,CS3247 | CS4350 | NM4260,CS4350 | NM4260 | CS3247 | HY2272 | ME5612,1,1,1.000000
8,working in a team to build a large software pr...,paraphrase,CS2103 | CS2103DE | CS2103R | CS2103T | CS2113...,MLE4102A | SWE5007 | CS3216 | CDE3301 | TCX3224,0,0,0.000000
9,how cryptocurrency and Bitcoin actually work,paraphrase,FIN4724 | IS4302,QF4211 | DSE4211 | CS4236 | FIN4724 | FT5003,4,1,0.250000


In [58]:
query = "Which NUS modules cover attention mechanisms?"

results = rag_utils.search(query, k=5)

print(f"Query: {query}\n")
print("Top 5 retrieved modules:\n")

for rank, (module, score) in enumerate(results, start=1):
    print(f"{rank}. {module.get('moduleCode', 'N/A')} - "
          f"{module.get('title', 'N/A')}")
    print(f"   Similarity score: {score:.4f}\n")

Query: Which NUS modules cover attention mechanisms?

Top 5 retrieved modules:

1. PL3281A - Lab in Perception and Attention
   Similarity score: 0.4792

2. NRM3302R - NUSC Undergraduate Research Opportunity (S&T UROP)
   Similarity score: 0.4097

3. GEX1009 - Understanding Your Brain
   Similarity score: 0.4058

4. NRM3301R - NUSC Undergraduate Research Opportunity (UROP)
   Similarity score: 0.4056

5. NRM3302 - NUSC Undergraduate Research Opportunity (S&T - UROP))
   Similarity score: 0.3839



In [59]:
query = input("Enter your question about NUS modules: ").strip()

if query:
    results = rag_utils.search(query, k=5)

    print(f"\nResults for: {query}\n")

    for rank, (module, score) in enumerate(results, start=1):
        print(f"{rank}. {module.get('moduleCode', 'N/A')} - "
              f"{module.get('title', 'N/A')}")
        print(f"   Similarity score: {score:.4f}\n")
else:
    print("Please enter a question.")


Results for: Escape

1. NTW2018 - Danger and Security
   Similarity score: 0.3010

2. GET1029 - Life, the Universe, and Everything
   Similarity score: 0.2739

3. NHS2096 - How to escape despotism?
   Similarity score: 0.2666

4. GL4886B - The International Refugee Regime
   Similarity score: 0.2607

5. SH5203C - Resilience and Emergency Planning
   Similarity score: 0.2571

